# MailSentinel email classification
Run from the project environment. This notebook uses the same loader and training function as the API release. No email content is printed.

In [1]:
from pathlib import Path
import sys, json
root = Path.cwd()
if root.name == 'ml':
    root = root.parent
sys.path.insert(0, str(root))
from ml.corpus import load
from ml.train import train, ARTIFACT

## Audit the corpus
Counts before and after deduplication reveal how the class mix changes. Missing subjects and bodies can overlap. A row with one usable field is retained.

In [2]:
texts, labels, eda = load()
for key in ('raw','clean','missing_subject','missing_body','duplicates','conflicting_groups','empty','median_chars','p95_chars'):
    print(key, eda[key])

raw {'legitimate': 4150, 'spam': 1896}
clean {'legitimate': 4107, 'spam': 1786}
missing_subject 19
missing_body 6
duplicates 153
conflicting_groups 0
empty 0
median_chars 1169
p95_chars 10877


## Fit and select
Set `retrain` to `True` to rebuild the release. The trainer fits TF-IDF inside each calibration fold, selects cutoffs and the release using false-positive-weighted validation cost, then evaluates all candidates on the archive test and short-message challenge.

In [3]:
retrain = False
report_file = ARTIFACT.with_name('metrics.json')
report = train() if retrain or not report_file.exists() else json.loads(report_file.read_text())
print('Split counts:', report['splits'])
for name, s in report['validation'].items():
    print(name, {k: round(s[k], 4) for k in ('accuracy','precision','recall','f1')}, s['confusion'])

Split counts: {'train': {'legitimate': 2903, 'spam': 1234}, 'validation': {'legitimate': 616, 'spam': 266}, 'short_validation': {'legitimate': 8, 'spam': 8}, 'test': {'legitimate': 616, 'spam': 268}, 'challenge': {'legitimate': 16, 'spam': 16}}
Multinomial Naive Bayes {'accuracy': 0.9807, 'precision': 0.9882, 'recall': 0.9474, 'f1': 0.9674} [[613, 3], [14, 252]]
Logistic Regression {'accuracy': 0.9864, 'precision': 0.9847, 'recall': 0.9699, 'f1': 0.9773} [[612, 4], [8, 258]]
Linear SVM {'accuracy': 0.9864, 'precision': 0.9811, 'recall': 0.9737, 'f1': 0.9774} [[611, 5], [7, 259]]
Random Forest {'accuracy': 0.9785, 'precision': 0.9625, 'recall': 0.9662, 'f1': 0.9644} [[606, 10], [9, 257]]


## Inspect the held-out result
Rows are actual classes and columns are predicted classes. Class order is legitimate, then spam. A false positive is a legitimate email classified as spam.

In [4]:
print('Selected:', report['winner'])
print('Held-out scores:', report['test'])

Selected: Multinomial Naive Bayes
Held-out scores: {'accuracy': 0.9717194570135747, 'precision': 0.9655172413793104, 'recall': 0.9402985074626866, 'f1': 0.9527410207939508, 'confusion': [[607, 9], [16, 252]], 'fpr': 0.01461038961038961, 'brier': 0.0236856133509271}
